# BeatTraffic KL — Crowd / Ridership Prediction v1

Roadmap item: **AI Crowd Prediction v1** — augments the client-side heuristic in
`src/lib/predictiveEngine.ts` with a model trained on real DOSM Rapid Rail data.

**Data source**: `Daily Origin-Destination Ridership: Rapid Rail (KV)`
(data.gov.my) — daily station-pair ridership across the entire Klang Valley
rail network, 2023–present. Schema: `date`, `origin`, `destination`, `ridership`.

**Known scope limit**: this dataset is daily frequency, not hourly. v1 therefore predicts
**daily per-station ridership volume**, and the existing heuristic's rush-hour/off-peak
multipliers are kept to shape that volume into a within-day crowd_score —
documented explicitly, not hidden.

**What's already done**: `ml/train_crowd_model.py` runs a synthetic LightGBM → ONNX
pipeline that the orchestration-api already serves. This notebook is the bridge to
real DOSM data so v2 can replace the synthetic training set.

## 0. Setup

In [ ]:
!pip install -q pandas pyarrow fastparquet lightgbm scikit-learn skl2onnx onnxmltools holidays matplotlib

import json
import math
import warnings
from pathlib import Path

import holidays
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from onnxmltools.convert.common.data_types import FloatTensorType
import onnxmltools
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', None)
REPO_ROOT = Path('..') if Path('../ml').exists() else Path('.')
ML_DIR = REPO_ROOT / 'ml'
MODELS_DIR = ML_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data acquisition

Pulling directly from data.gov.my's public storage — no API key needed.
Loop over available years. A HEAD check guards against silent empty-frame training.

In [ ]:
import urllib.request

YEARS = [2024, 2025, 2026]
BASE_URL = 'https://storage.data.gov.my/transportation/rail'

frames = []
for year in YEARS:
    url = f'{BASE_URL}/rapidrail_{year}_daily.parquet'
    # Validate the URL exists before downloading
    try:
        req = urllib.request.Request(url, method='HEAD')
        urllib.request.urlopen(req, timeout=5)
    except Exception as e:
        print(f'{year}: skipped (HEAD failed — {e})')
        continue
    try:
        df_year = pd.read_parquet(url)
        frames.append(df_year)
        print(f'{year}: {len(df_year):,} rows')
    except Exception as e:
        print(f'{year}: download failed ({e})')

if not frames:
    raise RuntimeError('No data loaded — check DOSM URL patterns and retry')

od = pd.concat(frames, ignore_index=True)
od['date'] = pd.to_datetime(od['date'])
print(f'\nTotal rows: {len(od):,}  |  Date range: {od["date"].min().date()} → {od["date"].max().date()}')
od.head()

## 2. Station → line mapping

Load from `ml/station_line_map.json` (committed to repo, derived from
`src/lib/transitData.ts` fallback stations + manual overrides).
Unmatched stations fall through to `None` — check the count before proceeding.

In [ ]:
with open(ML_DIR / 'station_line_map.json') as f:
    raw_map = json.load(f)
    station_line_map = {k: v for k, v in raw_map.items() if not k.startswith('_')}

print(f'station_line_map entries: {len(station_line_map)}')

all_stations = set(od['origin'].unique()) | set(od['destination'].unique())
matched = all_stations & set(station_line_map)
unmatched = all_stations - matched
coverage = len(matched) / len(all_stations) * 100 if all_stations else 0

print(f'Coverage: {len(matched)}/{len(all_stations)} stations ({coverage:.1f}%)')
print(f'{len(unmatched)} unmatched (add to station_line_map.json):')
print(sorted(unmatched)[:30])

## 3. Feature engineering (per station, per day)

In [ ]:
# Total daily volume per station = sum of trips where station is origin or destination
origin_vol = od.groupby(['date', 'origin'])['ridership'].sum().rename('outbound')
dest_vol   = od.groupby(['date', 'destination'])['ridership'].sum().rename('inbound')

daily = (
    pd.concat([origin_vol, dest_vol], axis=1)
    .fillna(0)
    .reset_index()
    .rename(columns={'origin': 'station'})
)
daily['total_volume'] = daily['outbound'] + daily['inbound']

# ── Calendar features ────────────────────────────────────────────────────────
daily['day_of_week'] = daily['date'].dt.dayofweek
daily['is_weekend']  = daily['day_of_week'].isin([5, 6]).astype(int)
daily['month']       = daily['date'].dt.month

# ── Malaysia public holidays via `holidays` package ──────────────────────────
MY_YEARS = daily['date'].dt.year.unique().tolist()
my_holidays = holidays.Malaysia(years=MY_YEARS, subdiv='KL')
daily['is_ph']      = daily['date'].isin(my_holidays).astype(int)
daily['is_eve_ph']  = daily['date'].shift(-1).isin(my_holidays).astype(int)  # day before PH

# ── Station metadata from line map ──────────────────────────────────────────
daily['line']           = daily['station'].map(lambda s: station_line_map.get(s, {}).get('line'))
daily['line_id']        = daily['station'].map(lambda s: station_line_map.get(s, {}).get('line_id', -1))
daily['zone']           = daily['station'].map(lambda s: station_line_map.get(s, {}).get('zone', 3))
daily['is_interchange'] = daily['station'].map(lambda s: station_line_map.get(s, {}).get('is_interchange', 0))

# ── Lag features — FIXED: assign back to daily per station group ─────────────
daily = daily.sort_values(['station', 'date'])
daily['lag_1d']          = daily.groupby('station')['total_volume'].shift(1)
daily['lag_7d']          = daily.groupby('station')['total_volume'].shift(7)
daily['rolling_7d_mean'] = daily.groupby('station')['total_volume'].transform(
    lambda x: x.shift(1).rolling(7, min_periods=1).mean()
)

print(f'daily shape: {daily.shape}')
print(f'NaN lag_1d rows: {daily["lag_1d"].isna().sum()} (expected ~n_stations, one per station cold-start)')
daily.head(10)

## 4. EDA — sanity-check against the existing heuristic's assumptions

In [ ]:
# predictiveEngine.ts assumes weekday >> weekend.
weekday_avg = daily[daily['is_weekend'] == 0]['total_volume'].mean()
weekend_avg = daily[daily['is_weekend'] == 1]['total_volume'].mean()
ph_avg      = daily[daily['is_ph'] == 1]['total_volume'].mean()
print(f'Weekday avg: {weekday_avg:.0f}  |  Weekend avg: {weekend_avg:.0f}  |  PH avg: {ph_avg:.0f}')
print(f'Weekday/Weekend ratio: {weekday_avg/weekend_avg:.2f}x')

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
daily.groupby('date')['total_volume'].sum().plot(ax=axes[0], title='Network-wide daily ridership')
daily.groupby('day_of_week')['total_volume'].mean().plot(
    kind='bar', ax=axes[1], title='Avg volume by day of week (0=Mon)')
plt.tight_layout()
plt.show()

## 5. Baseline models

Two baselines to beat before the LightGBM model counts as a real improvement:
1. **Naive persistence** — yesterday's value
2. **7-day rolling mean** — smooths day-of-week noise

If LightGBM doesn't clear both by a meaningful margin, that's a legitimate
finding to report, not a failure to hide.

In [ ]:
FEATURES = ['day_of_week', 'is_weekend', 'month', 'is_ph', 'is_eve_ph',
            'is_interchange', 'zone', 'lag_1d', 'lag_7d', 'rolling_7d_mean']
TARGET = 'total_volume'

# Drop stations not in line map and rows with NaN lags
df_model = daily[daily['line_id'] >= 0].dropna(subset=FEATURES + [TARGET]).copy()
print(f'Training rows: {len(df_model):,}')

# Time-ordered split — no shuffle to prevent look-ahead leakage
split_date = df_model['date'].quantile(0.8)
X_train = df_model[df_model['date'] <= split_date][FEATURES]
y_train = df_model[df_model['date'] <= split_date][TARGET]
X_test  = df_model[df_model['date'] >  split_date][FEATURES]
y_test  = df_model[df_model['date'] >  split_date][TARGET]
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                           num_leaves=63, min_child_samples=30, verbose=-1)
model.fit(X_train, y_train)

pred               = model.predict(X_test)
baseline_persist   = X_test['lag_1d']
baseline_rolling   = X_test['rolling_7d_mean']

for name, p in [('LightGBM', pred), ('Persistence', baseline_persist), ('7d rolling', baseline_rolling)]:
    mae  = mean_absolute_error(y_test, p)
    mape = mean_absolute_percentage_error(y_test, p)
    print(f'{name:12s}  MAE={mae:8.1f}  MAPE={mape:.2%}')

## 6. Per-line evaluation

Break out MAE by line — the model should beat baselines on high-traffic lines
(Kelana Jaya, Kajang) even if it's weaker on low-volume ones (Monorail).
Report both, don't average them away.

In [ ]:
test_df = df_model[df_model['date'] > split_date].copy()
test_df['pred'] = pred
test_df['persist'] = X_test['lag_1d'].values

per_line = []
for line, grp in test_df.groupby('line'):
    mae_lgb  = mean_absolute_error(grp[TARGET], grp['pred'])
    mae_base = mean_absolute_error(grp[TARGET], grp['persist'])
    per_line.append({'line': line, 'n': len(grp), 'MAE_LGB': mae_lgb, 'MAE_persist': mae_base,
                     'improvement_%': (mae_base - mae_lgb) / mae_base * 100})

results_df = pd.DataFrame(per_line).sort_values('improvement_%', ascending=False)
print(results_df.to_string(index=False, float_format='{:.1f}'.format))

## 7. Export

Export to ONNX so `orchestration-api` can load it directly.
**Note**: this is a *regressor* (daily volume), not the 3-class classifier in
`train_crowd_model.py`. The API endpoint maps the volume percentile back to
0/1/2 crowd_level at inference time.

In [ ]:
assert len(FEATURES) == X_train.shape[1], f'Feature count mismatch: {len(FEATURES)} vs {X_train.shape[1]}'

initial_type = [('float_input', FloatTensorType([None, len(FEATURES)]))]
onnx_model = onnxmltools.convert_lightgbm(model, initial_types=initial_type, target_opset=15)

onnx_path = MODELS_DIR / 'crowd_model_v1_dosm.onnx'
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())
print(f'ONNX model saved → {onnx_path}')

# Write metrics.json alongside the model
overall_mae  = mean_absolute_error(y_test, pred)
overall_mape = mean_absolute_percentage_error(y_test, pred)
per_line_mae = results_df.set_index('line')['MAE_LGB'].to_dict()

metrics = {
    'version': 'v1-dosm',
    'data_source': 'DOSM Daily OD Ridership: Rapid Rail (KV)',
    'features': FEATURES,
    'n_features': len(FEATURES),
    'task': 'regression (daily total volume per station)',
    'overall_mae': round(overall_mae, 1),
    'overall_mape': round(overall_mape, 4),
    'per_line_mae': {k: round(v, 1) for k, v in per_line_mae.items()},
}
metrics_path = MODELS_DIR / 'metrics_v1_dosm.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f'Metrics saved → {metrics_path}')
print(json.dumps(metrics, indent=2))

## 8. Next steps

1. **Fill `ml/station_line_map.json`** with all unmatched DOSM station names
   (run cell 6 and add each name to the JSON).
2. **Wire `crowd_model_v1_dosm.onnx`** into a new route in
   `orchestration-api/app/api/routes/crowd_prediction.py` — the v1 endpoint
   takes `{station_name, date}` and returns `{daily_volume, crowd_level}`.
3. **Update `predictiveEngine.ts`** to call the daily-volume endpoint and
   combine it with the existing intraday heuristic multipliers.
4. **Replace synthetic training** in `ml/train_crowd_model.py` by feeding
   real ridership counts from this notebook's `daily` DataFrame back in as
   the tap-count proxy (`prev_tap_t_15` / `prev_tap_t_30`).